# Qwen3.5-0.8B LoRA Fine-Tuning - Colab
Fine-tune Qwen3.5-0.8B on any dataset from michaelowusuntim6 on HF.
Requirements: Colab T4 (16 GB VRAM). bf16 LoRA uses ~3 GB VRAM.
Do NOT use QLoRA 4-bit - Qwen3.5 is not recommended for it.


In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps transformers trl peft accelerate bitsandbytes datasets
!pip install -q --no-deps unsloth_zoo


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

DATASET_NAME = "code-review-qwen35"  # change to any of the 30
MAX_SEQ_LEN = 8192
EXPERT_NAME = "code_review"


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-0.8B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,           # auto (bf16 on T4)
    load_in_4bit=False,   # QLoRA not recommended for Qwen3.5
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


In [ ]:
ds = load_dataset(f"michaelowusuntim6/{DATASET_NAME}", split="train")

# Pre-format with the model's native chat template
def format_record(ex):
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return ex

ds = ds.map(format_record, num_proc=2)


In [ ]:

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        output_dir=f"outputs/{EXPERT_NAME}",
        save_strategy="epoch",
        report_to="none",
    ),
)
trainer.train()


In [ ]:
model.save_pretrained(f"adapters/{EXPERT_NAME}")
tokenizer.save_pretrained(f"adapters/{EXPERT_NAME}")


In [ ]:
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])
# model.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}-adapter")
# tokenizer.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}-adapter")


In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "How do I fix a memory leak in C?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")
out = model.generate(inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
